# Fairness Preprocessing Tutorial

This tutorial demonstrates how to apply the **fairness preprocessing** method in PipeGenie before training a classification model.

In this example, we use the **Employee Promotion** dataset and evaluate the model using both performance and fairness metrics. 

## 1. Load and Prepare the Dataset

First, we load the dataset, define:

- **X**: input features
- **y**: target variable (`is_promoted`)
- **A**: sensitive feature (`education`)

Then we split the dataset into training and testing sets.

In [1]:
import pandas as pd

df = pd.read_csv("../datasets/employee_promotion.csv")

y = df["is_promoted"]

df["education"] = df["education"].fillna("Unknown")
A = df["education"]

X = df.drop(columns=["is_promoted","education"])
X = pd.get_dummies(X, drop_first=True)

In [2]:
from pipegenie.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25,random_state=42)
A_train, A_test, _, _ = train_test_split(A,y,test_size=0.25,random_state=42)

## 2. Apply Fairness Preprocessing

The preprocessing step modifies the training data before the model is trained.

Its goal is to reduce bias with respect to the selected sensitive feature while preserving as much predictive performance as possible.

In [3]:
from pipegenie.fairness.preprocessing import fairness_preprocessing

X_train_fair, y_train_fair = fairness_preprocessing(X_train,y_train,A_train)

## 3. Create the PipeGenie Classifier

Next, we initialize a `PipegenieClassifier` using the classification grammar.

The grammar defines the search space that PipeGenie explores to automatically build machine learning pipelines.

In [4]:
grammar_file = "../../grammars/sample-grammar-classification.xml"

from pipegenie.classification import PipegenieClassifier
model = PipegenieClassifier(
        grammar=grammar_file,
        generations=5,
        pop_size=50,
        elite_size=5,
        seed=42,
        outdir="samples/classification_pre")

## 4. Train the Model

The model is trained using the fairness-preprocessed training set.

In [5]:
model.fit(X_train_fair, y_train_fair)

                                       fitness                                            size                                        fitness_elite                                      size_elite              
                 ----------------------------------------------------    --------------------------------------    ----------------------------------------------------    --------------------------------------
gen    nevals    min           max           avg           std           min    max    avg           std           min           max           avg           std           min    max    avg           std       
0      50        0.53773       0.91796       0.79515       0.15046       1      5      2.64          1.3381        0.53773       0.91796       0.79515       0.15046       1      3      1.5           0.86603   
1      50        0.53773       0.91796       0.79444       0.13458       1      5      2.7           1.3304        0.53773       0.91796       0.79444       0.1

## 5. Generate Predictions

After training, predictions are generated on the test set.

In [6]:
y_pred = model.predict(X_test)

## 6. Evaluate Fairness

The fairness report computes:

- Overall model performance
- Performance for each sensitive group

This allows us to inspect whether the model behaves differently across groups.

In [7]:
from pipegenie.fairness import fairness_report
from pipegenie.metrics import balanced_accuracy_score

report = fairness_report( y_true=y_test, y_pred=y_pred, sensitive_features=A_test, metric=balanced_accuracy_score)

print("\n==============================")
print("OVERALL PERFORMANCE")
print("==============================")

print(report["overall"])

print("\n==============================")
print("PERFORMANCE BY GROUP")
print("==============================")

print(report["by_group"])


OVERALL PERFORMANCE
0.6725188606008042

PERFORMANCE BY GROUP
sensitive_feature_0
Bachelors          0.669272
Below Secondary    0.652216
Mastersabove       0.675637
Unknown            0.715109
Name: balanced_accuracy_score, dtype: float64


## 7. Compute Standard Fairness Metrics

Finally, we compute commonly used fairness metrics:

- **Balanced Accuracy**
- **Equalized Odds Difference**
- **Demographic Parity Difference**

These metrics provide a quantitative evaluation of fairness after preprocessing.

In [8]:
# =====================================================
# STANDARD METRICS
# =====================================================

from fairlearn.metrics import (equalized_odds_difference,demographic_parity_difference)


print("\n==============================")
print("BALANCED ACCURACY")
print("==============================")

print( balanced_accuracy_score(y_test, y_pred))

eo = equalized_odds_difference(y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

dp = demographic_parity_difference(y_true=y_test, y_pred=y_pred, sensitive_features=A_test)

print("\n===== FAIRNESS METRICS =====")
print("Equalized Odds Difference:", eo)
print("Demographic Parity Difference:", dp)


BALANCED ACCURACY
0.6725188606008042

===== FAIRNESS METRICS =====
Equalized Odds Difference: 0.13541666666666669
Demographic Parity Difference: 0.03912968843876063


## Conclusion

In this tutorial, we:

- Loaded a classification dataset.
- Applied the fairness preprocessing method.
- Trained a PipeGenie classifier.
- Generated predictions.
- Evaluated both predictive performance and fairness.

This workflow demonstrates how fairness preprocessing can be integrated into a standard PipeGenie classification pipeline.